In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification
)

/home/msmith/apps/comhaireachd/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = AutoModelForTokenClassification.from_pretrained(
    'Synthyra/ESMplusplus_small',
    trust_remote_code=True,
    num_labels=3
)
tokenizer = model.tokenizer

Some weights of ESMplusplusForTokenClassification were not initialized from the model checkpoint at Synthyra/ESMplusplus_small and are newly initialized: ['classifier.0.bias', 'classifier.0.weight', 'classifier.2.bias', 'classifier.2.weight', 'classifier.3.bias', 'classifier.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## TMbed training data

Gonna use the training data from TMbed - specifically the alpha helix and globular signal peptide annotated fasta files.
I need to get these into a csv file of gene ID, sequence, and associated labels.
For now, let's ignore the inside/outside status - I'm only really interested in the signal peptides and alpha-helix annotations.
We will say that:

```
0 = non-annotated
1 = signal peptide
2 = alpha helix
```

In [3]:
def replace_labels(label):
    mapping = {
        "U": 0,
        "1": 0,
        "2": 0,
        "S": 1,
        "H": 2,
        "h": 2,
    }

    new_label = np.zeros(len(label), dtype=int)
    for i, char in enumerate(label):
        if char in mapping:
            new_label[i] = mapping[char]
        else:
            raise ValueError(f"Unexpected label character: {char}")
    return new_label
        

data = {
    "id": [],
    "sequences": [],
    "labels": [],
}

with open("data/tmbed-helix.fa") as f:
    position = 0 # fasta is in three line format, so need to track
    for line in f:
        line = line.strip()
        if position == 0:
            id = line.split("|")[0][1:] # remove > and description stuff
            data["id"].append(id)
            position += 1
        elif position == 1:
            data["sequences"].append(line)
            position += 1
        elif position == 2:
            labels = replace_labels(line)
            data["labels"].append(labels)
            position = 0

with open("data/tmbed-signal.fa") as f:
    position = 0 # fasta is in three line format, so need to track
    for line in f:
        line = line.strip()
        if position == 0:
            id = line.split("|")[0][1:] # remove > and description stuff
            data["id"].append(id)
            position += 1
        elif position == 1:
            data["sequences"].append(line)
            position += 1
        elif position == 2:
            labels = replace_labels(line)
            data["labels"].append(labels)
            position = 0


In [4]:
train_sequences, test_sequences, train_labels, test_labels = train_test_split(data["sequences"], data["labels"], test_size=0.25, shuffle=True)

train_tokenized = tokenizer(train_sequences)
test_tokenized = tokenizer(test_sequences)

train_dataset = Dataset.from_dict(train_tokenized)
test_dataset = Dataset.from_dict(test_tokenized)

train_dataset = train_dataset.add_column("labels", train_labels)
test_dataset = test_dataset.add_column("labels", test_labels)

In [5]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [7]:
training_args = TrainingArguments(
    output_dir="tm_model",
    num_train_epochs=20,
    gradient_accumulation_steps=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    logging_dir="tm_model",
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    label_names=["labels"]
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
)

trainer.train()

/home/msmith/apps/comhaireachd/.venv/lib/python3.13/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.835800,0.360143
2,0.228000,0.160921
3,0.137700,0.144388
4,0.098400,0.149933
5,0.075800,0.158311
6,0.077600,0.174254
7,0.070800,0.183129
8,0.071800,0.178521
9,0.050900,0.204764
10,0.060600,0.197265


/home/msmith/apps/comhaireachd/.venv/lib/python3.13/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/msmith/apps/comhaireachd/.venv/lib/python3.13/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/msmith/apps/comhaireachd/.venv/lib/python3.13/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/msmith/apps/comhaireachd/.venv/lib/python3.13/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.wa

KeyboardInterrupt: 

In [8]:
model = AutoModelForTokenClassification.from_pretrained("tm_model/checkpoint-224", trust_remote_code=True, num_labels=3)
tokenizer = model.tokenizer

In [12]:
for i in range(len(data["id"])):
    id = data["id"][i]
    sequence = data["sequences"][i]
    labels = data["labels"][i]
    
    inputs = tokenizer(sequence, return_tensors="pt")
    outputs = model(**inputs)
    predictions = outputs.logits.argmax(dim=-1).squeeze().tolist()
    print(predictions)


[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 